# TP4 : Modélisation Bayésienne, Bootstrap et Théorie de la Décision

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu
1. Travail individuel. Renommez le fichier `TP4_Nom_Prenom.ipynb`.
2. Votre code doit s'exécuter de bout en bout sans erreur (Kernel > Restart & Run All).
3. Remplacez les `...` par votre code et répondez aux questions textuelles dans les blocs **Analyse**.

---

## Partie 1 : Modélisation Bayésienne Naïve (Exoplanètes - Kepler)

### Rappels Mathématiques (Extrait du TD4)

Nous cherchons à classifier un objet céleste $X$ en deux classes : Exoplanète ($E$, cible=1) ou Naine Brune/Anomalie ($N$, cible=0). L'algorithme de Bayes Naïf suppose l'indépendance conditionnelle des caractéristiques $x_1, x_2, x_3$ sachant la classe $C \in \{E, N\}$ :

$$ P(C \mid x_1, x_2, x_3) \propto P(C) \prod_{i=1}^{3} P(x_i \mid C) $$

Afin d'éviter les problèmes d'underflow numérique, nous travaillerons avec les **log-vraisemblances** :

$$ \log P(C \mid X) \propto \log P(C) + \sum_{i=1}^{3} \log P(x_i \mid C) $$

Les lois conditionnelles modélisant nos variables sont :
1. **Bernoulli (Éclipse)** : $P(X=x) = p^x (1-p)^{(1-x)}$ pour $x \in \{0, 1\}$
2. **Poisson (Durée)** : $P(X=x) = \frac{\lambda^x e^{-\lambda}}{x!}$ pour $x \in \mathbb{N}$
3. **Gaussienne (Température)** : $f(x) = \frac{1}{\sigma \sqrt{2\pi}} \exp{\left(-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2\right)}$ pour $x \in \mathbb{R}$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import gammaln

sns.set_theme(style="whitegrid")

### 1.1 Préparation des variables guidée par la physique

Chargez le fichier `cumulative.csv` depuis [Kepler Exoplanet Search Results](https://www.kaggle.com/datasets/nasa/kepler-exoplanet-search-results?resource=download). Pour correspondre à notre cadre théorique, nous devons écarter les objets incertains (`CANDIDATE`).

In [ ]:
# Chargement des données
df = pd.read_csv('cumulative.csv')

# Filtrage : Conservez uniquement les lignes où koi_disposition n'est PAS 'CANDIDATE'
df = df[df[...] != ...].copy()

# Création de la variable cible binaire : 1 si 'CONFIRMED', 0 si 'FALSE POSITIVE'
df['target'] = (df['koi_disposition'] == ...).astype(int)

Pour modéliser $x_1$ (Bernoulli), nous devons binariser la profondeur de l'éclipse (`koi_depth`). Plutôt que de choisir un seuil arbitraire, utilisez un arbre de décision de profondeur 1 pour trouver le seuil optimal séparant les deux classes.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Remplacement des NaN par la médiane
depth = df['koi_depth'].fillna(df['koi_depth'].median()).values.reshape(-1, 1)

# Entraînez un arbre de décision avec max_depth=1
tree = DecisionTreeClassifier(max_depth=1, random_state=42)
tree.fit(..., ...)

# Extraction du seuil optimal trouvé par l'arbre
threshold_x1 = tree.tree_.threshold[0]
print(f"Seuil optimal pour x1 : {threshold_x1:.2f} ppm")

# Binarisation : x1 = 1 si la profondeur dépasse le seuil, 0 sinon
df['x1_eclipse'] = (depth.flatten() > ...).astype(int)

Pour $x_2$ (Poisson, comptage), nous utiliserons la durée du transit en heures (`koi_duration`), que nous arrondirons à l'entier le plus proche.

Pour $x_3$ (Gaussienne), la température `koi_teq` est très asymétrique (skewed). Nous appliquerons la fonction logarithme pour la "Gaussianiser".

In [ ]:
# Traitement de x2 (Poisson) : Remplissez les NaN par la médiane puis arrondissez
duree = df['koi_duration'].fillna(df['koi_duration'].median())
df['x2_duree'] = np.round(...).astype(int)

# Traitement de x3 (Gaussienne) : Remplissez les NaN par la médiane puis appliquez np.log(x + 1)
teq = df['koi_teq'].fillna(df['koi_teq'].median())
df['x3_temp'] = np.log(...)

Afin de valider visuellement nos transformations, traçons les distributions de nos trois variables. Observez comment elles se rapprochent des lois théoriques attendues (Bernoulli, Poisson, Gaussienne).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# x1 : Bernoulli (Countplot)
sns.countplot(data=df, x='x1_eclipse', hue='target', ax=axes[0])
axes[0].set_title("x1 : Éclipse Secondaire")

# x2 : Poisson (Histplot pour données de comptage)
sns.histplot(data=df, x='x2_duree', hue='target', binwidth=1, ax=axes[1])
axes[1].set_xlim(0, 20)
axes[1].set_title("x2 : Durée du transit (Heures)")

# x3 : Gaussienne (KDE plot pour distribution continue)
sns.kdeplot(data=df, x='x3_temp', hue='target', common_norm=False, ax=axes[2])
axes[2].set_title("x3 : Log(Température)")

plt.tight_layout()
plt.show()

### 1.2 Vérification de l'hypothèse de Bayes Naïf

L'algorithme suppose l'indépendance conditionnelle. Affichez la matrice de corrélation de Spearman entre $x_1, x_2, x_3$ *pour chaque classe séparément*.

In [ ]:
features = ['x1_eclipse', 'x2_duree', 'x3_temp']

print("Corrélations pour les Exoplanètes (Cible = 1) :")
display(df[df['target'] == 1][features].corr(method='spearman'))

print("\nCorrélations pour les Naines/Anomalies (Cible = 0) :")
display(df[df['target'] == 0][features]...)

**Analyse :** Double-cliquez ici. L'hypothèse mathématique de Bayes Naïf est-elle respectée sur nos données astrophysiques ? Justifiez.

### 1.3 Estimation du Maximum de Vraisemblance (MLE)

Vous devez estimer les paramètres théoriques pour chaque classe à partir des données : $p$ (moyenne pour Bernoulli), $\lambda$ (moyenne pour Poisson), $\mu$ et $\sigma$ (moyenne et écart-type pour Gauss).

In [ ]:
def fit_mle(df, target_class):
    """Calcule les paramètres MLE pour une classe donnée."""
    sub_df = df[df['target'] == target_class]
    
    params = {
        'p': sub_df['x1_eclipse']...(),
        'lambda': sub_df['x2_duree']...(),
        'mu': sub_df['x3_temp']...(),
        'sigma': sub_df['x3_temp']...()
    }
    return params

params_E = fit_mle(df, 1)
params_N = fit_mle(df, 0)

print("Paramètres Exoplanètes :", params_E)
print("Paramètres Anomalies :", params_N)

### 1.4 Implémentation mathématique de la Log-Vraisemblance

Traduisez les formules du rappel mathématique en code vectorisé NumPy. 
*Astuce numérique* : Pour $\log(x!)$ de la loi de Poisson, utilisez la fonction `gammaln(x + 1)` de scipy pour éviter le dépassement de capacité (overflow).

In [ ]:
def compute_log_likelihood(X, params):
    """
    Calcule la somme des log-vraisemblances pour les 3 features.
    X est une matrice NumPy de forme (n_samples, 3).
    """
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    
    # Sécurisation numérique (éviter log(0))
    p = np.clip(params['p'], 1e-9, 1-1e-9)
    lam = max(params['lambda'], 1e-9)
    mu, sigma = params['mu'], max(params['sigma'], 1e-9)
    
    # 1. Log-vraisemblance Bernoulli (x1)
    ll_x1 = x1 * np.log(p) + (1 - x1) * np.log(...)
    
    # 2. Log-vraisemblance Poisson (x2)
    ll_x2 = x2 * ... - ... - gammaln(...)
    
    # 3. Log-vraisemblance Gaussienne (x3)
    ll_x3 = ...
    
    # Hypothèse Naïve : P(x1,x2,x3|C) = P(x1|C)*P(x2|C)*P(x3|C)
    # En log, le produit devient une somme.
    return ...

Appliquez le théorème de Bayes et la fonction **Softmax** pour calculer la probabilité a posteriori.

In [ ]:
def predict_proba_bayes(X, params_E, params_N, prior_E):
    prior_N = 1.0 - prior_E
    
    # Calcul des log-probabilités non normalisées (Vraisemblance + Log-Apriori)
    log_prob_E = compute_log_likelihood(X, params_E) + np.log(...)
    log_prob_N = compute_log_likelihood(X, params_N) + np.log(...)
    
    # Application du Softmax avec astuce numérique log-sum-exp
    max_log = np.maximum(log_prob_E, log_prob_N)
    exp_E = np.exp(log_prob_E - max_log)
    exp_N = np.exp(log_prob_N - max_log)
    
    # Voir la correction de question 4, exercise 3, TD 4
    return ...

En apprentissage automatique, on évalue généralement les modèles sur un ensemble de test distinct. Cependant, notre objectif étant d'estimer les paramètres physiques réels de l'univers avec une précision maximale, nous calculons l'estimateur du maximum de vraisemblance (EMV) et l'évaluons sur l'ensemble des données. Nous appliquerons une division rigoureuse des données d'entraînement et de test dans la partie 2.

In [ ]:
# Prédiction sur le jeu entier
X_kep = df[['x1_eclipse', 'x2_duree', 'x3_temp']].values
y_kep = df['target'].values

# Calcul des probabilités a priori depuis le jeu de données
prior_E = df['target']....()
prior_N = 1.0 - prior_E

print(f"Prior Exoplanète P(E) : {prior_E:.3f}")
print(f"Prior Anomalie P(N) : {prior_N:.3f}")

y_prob_kep = predict_proba_bayes(X_kep, params_E, params_N, prior_E)

Calculez et affichez l'AUC (Area Under the ROC Curve) de votre modèle mathématique en utilisant la librairie `sklearn.metrics`.

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(..., ...)
auc_score = auc(fpr, tpr)

print(f"AUC de notre modèle Naïve Bayes : {auc_score:.3f}")

---

### 1.5 Questions de réflexion

**Question 1 : L'impact de la probabilité a priori (Prior)**
Dans notre calcul, nous avons utilisé les probabilités a priori empiriques du jeu de données. Si vous modifiez le code pour forcer $P(E) = 0.2$ et $P(N) = 0.8$ (comme dans le TD), constaterez-vous le changement d'AUC ? 
Codez et donnez votre explication.

**Analyse (Q1) :** (Indice : réfléchissez à ce que mesure la courbe ROC par rapport au classement/rang des prédictions).

**Question 2 : Amélioration de la performance**
En considérant les limites de l'algorithme Naïve Bayes et la manière dont nous avons préparé les données (binarisation, choix des lois mathématiques...), proposez **deux stratégies distinctes** qui permettraient d'augmenter significativement cette AUC.

**Analyse (Q2) :** Double-cliquez ici pour répondre. 
- Stratégie 1 : ...
- Stratégie 2 : ...

## Partie 2 : Intervalles de Confiance & Bootstrap (Credit-g)

### Rappels Mathématiques
Contrairement à la Régression Linéaire par Moindres Carrés (OLS) où la variance des résidus suit une loi de Student exacte (TD 5), la Régression Logistique ne possède pas de distribution exacte pour de petits échantillons.

Pour estimer la **variance de notre modèle** (Incertitude Épistémique), nous utilisons le **Bootstrap** :
1. Tirer $B$ échantillons avec remise de taille $N$ depuis les données d'apprentissage.
2. Ajuster le modèle sur chaque échantillon b.
3. Pour une nouvelle observation $x$, obtenir $B$ prédictions $\hat{p}_b(x)$.
4. L'intervalle de confiance à 95% est délimité par les centiles 2.5% et 97.5% de ces $B$ prédictions.

### 2.1 Préparation du jeu de données
Téléchargez le jeu de données `credit-g`. Construisez un pipeline avec `StandardScaler` pour les variables numériques et `OneHotEncoder` pour les catégories.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

Le data_id de `credit-g` est 31: https://www.openml.org/search?type=data&status=active&id=31

In [ ]:
data_credit = fetch_openml(data_id=..., as_frame=True, parser='auto')
df_cred = data_credit.frame

Utilisez `df.info` et `df.head(...)` pour déterminer `y_cred`.

In [ ]:
...
y_cred = (df_cred['class'] == ...).astype(int).values
X_cred = df_cred.drop(columns=['class'])

Consultez le [guide](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html) et la [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) de `ColumnTransformer` pour la cellule suivante.

In [ ]:
# Séparation Train/Test AVANT prétraitement (Évite le Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X_cred, y_cred, test_size=0.3, random_state=42)

numeric_cols = X_train.select_dtypes(include=['number']).columns
categorical_cols = X_train.select_dtypes(exclude=['number']).columns

# Complétez le ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', ..., numeric_cols), # StandardScaler
        ('cat', OneHotEncoder(drop='first', sparse_output=False), ...)
    ])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

### 2.2 Évaluation du modèle de base et Courbe ROC

Avant d'estimer l'incertitude via Bootstrap, entraînons un modèle de base unique pour évaluer ses performances globales. Nous utiliserons la fonction `roc_curve` de `sklearn.metrics`.

**Rappel sur la fonction `roc_curve` :**
Cette fonction prend en entrée les vrais labels (`y_true`) et les probabilités prédites (`y_score`). Elle retourne **3 tableaux (arrays)** fondamentaux que vous réutiliserez pour la Théorie de la Décision (Partie 3) :
- **`fpr`** (False Positive Rate) : Le taux de Faux Positifs pour différents seuils.
- **`tpr`** (True Positive Rate) : Le taux de Vrais Positifs (Rappel) pour différents seuils.
- **`thresholds`** : Les seuils de probabilité associés. Un point se trouvant à l'indice $i$ des tableaux `fpr` et `tpr` correspond aux performances du modèle si l'on décide que la classe est 1 uniquement lorsque $P(Y=1 \mid X) \ge \text{thresholds}[i]$.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc

# Entraînement du modèle de base
model_base = LogisticRegression(max_iter=1000)
model_base.fit(..., ...)

# Prédiction des probabilités pour la classe 1 (good credit)
y_prob_base = model_base.predict_proba(...)[:, 1]

# Extraction des taux et seuils via roc_curve
fpr, tpr, thresholds = roc_curve(..., ...)
roc_auc = auc(fpr, tpr)

print(f"AUC du modèle de base : {roc_auc:.3f}")

In [ ]:
plt.figure(figsize=(6, 5))

# Tracé de la courbe ROC
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Courbe ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs (TPR)')
plt.title('Courbe ROC - Régression Logistique (Credit-g)')
plt.legend(loc="lower right")
plt.show()

### 2.3 Implémentation du Bootstrap
Codez la boucle Bootstrap pour générer 100 modèles de régression logistique et extraire l'intervalle de confiance (IC) pour chaque point de `X_test_proc`.

In [ ]:
from sklearn.linear_model import LogisticRegression

n_bootstraps = 100
n_samples = X_train_proc.shape[0]
preds_boot = np.zeros((n_bootstraps, X_test_proc.shape[0]))

for i in range(n_bootstraps):
    # 1. Tirage avec remise des indices d'entraînement
    indices = np.random.choice(n_samples, size=..., replace=...)
    X_boot, y_boot = X_train_proc[indices], y_train[indices]
    
    # 2. Ajustement du modèle
    model = LogisticRegression(max_iter=1000)
    model.fit(..., ...)
    
    # 3. Prédiction des probabilités sur X_test_proc
    preds_boot[i, :] = model.predict_proba(...)[:, 1]

# Calcul de la moyenne et des bornes de l'IC à 95% via np.percentile
mean_prob = np.mean(preds_boot, axis=0)
ci_lower = np.percentile(preds_boot, q=..., axis=0)
ci_upper = np.percentile(preds_boot, q=..., axis=0)

### 2.4 Analyse de l'Incertitude

En Machine Learning, lorsqu'un modèle est incertain (il prédit une probabilité proche de 0.5), cette incertitude peut avoir deux origines fondamentales qu'il est crucial de distinguer :

1. **L'incertitude Aléatorique (Le bruit inhérent)** : 
   C'est l'incertitude due au hasard du monde réel ou au bruit dans les données. Même avec un modèle parfait et une infinité de données, on ne pourrait pas trancher. 
   *Exemple :* Deux clients ont le *même* profil exact, mais l'un rembourse son crédit et l'autre non suite à un accident de la vie imprévisible.
   *Symptôme Bootstrap :* Tous les modèles générés sont d'accord pour dire qu'ils ne savent pas. L'intervalle de confiance est **étroit** autour de 0.5 (ex: `[0.48, 0.52]`).

2. **L'incertitude Épistémique (L'ignorance du modèle)** : 
   C'est l'incertitude due à un manque de connaissances. Le modèle fait face à une donnée aberrante ou inédite (*Out-Of-Distribution* - OOD) qu'il n'a jamais vue lors de son entraînement. S'il avait plus de données similaires, il saurait répondre.
   *Symptôme Bootstrap :* Comme la région de l'espace est inconnue, les petits changements dans les données d'entraînement (causés par le tirage avec remise) font complètement basculer la décision d'un modèle à l'autre. Le désaccord est total, l'intervalle de confiance est **très large** (ex: `[0.10, 0.90]`).

Exécutez la cellule ci-dessous pour isoler le point de test qui génère le plus grand désaccord parmi vos 100 modèles Bootstrap, et analysez ses caractéristiques.

In [ ]:
ci_width = ci_upper - ci_lower
idx_max_var = np.argmax(ci_width)

print(f"Le profil n°{idx_max_var} génère un désaccord massif entre les modèles Bootstrap.")
print(f"Probabilité moyenne : {mean_prob[idx_max_var]:.2f}")
print(f"Intervalle de Confiance (95%) : [{ci_lower[idx_max_var]:.2f}, {ci_upper[idx_max_var]:.2f}]")
print("\nCaractéristiques originales du client :")
display(X_test.iloc[idx_max_var])

**Analyse :** Observez la probabilité moyenne et l'intervalle de confiance du client affiché ci-dessus. 
Au vu des caractéristiques originales du client (montant du crédit, durée, etc.), ce profil vous semble-t-il classique ? Quel type d'incertitude (Aléatorique ou Épistémique) est mis en évidence ici par le Bootstrap ? Dans un système bancaire réel, quelle décision métier prendriez-vous face à ce dossier spécifique ?

---

### 2.5 Régions de Prédiction et Performance

Contrairement à la régression où l'on ajoute $\sigma^2$ pour obtenir un intervalle de prédiction, en classification, la **région de prédiction** est l'ensemble des classes plausibles. Si l'intervalle de confiance de la probabilité chevauche notre seuil de décision (0.5), cela signifie que les classes 0 et 1 sont toutes les deux plausibles : **le modèle hésite**.

Nous allons faire varier le niveau de confiance exigé de notre Bootstrap (de 40% à 98%). Plus on exige une grande confiance, plus les intervalles (Variance) s'élargissent, et plus le modèle rejettera d'échantillons dans sa région d'hésitation. 
Complétez le code ci-dessous pour observer l'impact de ce rejet sur l'Accuracy des clients restants (les prédictions "sûres").

In [ ]:
# Prédiction finale moyenne de base (seuil 0.5)
y_pred_final = (mean_prob >= 0.5).astype(int)

confidence_levels = np.linspace(0.40, 0.98, 50)
accuracies_on_sure = []
hesitation_rates = []

for conf in confidence_levels:
    # Calcul des quantiles (ex: conf=0.90 -> q_low=5%, q_high=95%)
    alpha = 1.0 - conf
    q_low = (alpha / 2.0) * 100
    q_high = (1.0 - alpha / 2.0) * 100
    
    ci_lower = np.percentile(preds_boot, q=..., axis=0)
    ci_upper = np.percentile(preds_boot, q=..., axis=0)
    
    # Masque d'hésitation : le seuil de 0.5 est à l'intérieur de l'intervalle
    hesitating_mask = (ci_lower < ...) & (ci_upper > ...)
    
    # Calcul du taux de rejet (hésitation)
    hesitation_rate = np.sum(hesitating_mask) / len(y_test)
    hesitation_rates.append(...)
    
    # Calcul de l'accuracy uniquement sur la partie "sûre" (~hesitating_mask)
    sure_mask = ~hesitating_mask
    if np.sum(sure_mask) > 0:
        acc = np.mean(y_pred_final[...] == y_test[...])
    else:
        acc = np.nan # Si le modèle doute de tout
        
    accuracies_on_sure.append(acc)

# Tracé du graphique à double axe
fig, ax1 = plt.subplots(figsize=(9, 5))

color1 = 'tab:blue'
ax1.set_xlabel('Niveau de confiance exigé du Bootstrap (%)')
ax1.set_ylabel('Accuracy sur les prédictions "Sûres"', color=color1)
ax1.plot(confidence_levels * 100, accuracies_on_sure, color=color1, lw=2)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0.5, 1.0)

ax2 = ax1.twinx()  
color2 = 'tab:red'
ax2.set_ylabel('Pourcentage d\'échantillons incertains (région $\{0, 1\}$) (Hésitation)', color=color2)
ax2.plot(confidence_levels * 100, np.array(hesitation_rates) * 100, color=color2, lw=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(0, 100)

plt.title("Performance vs Incertitude (Régions de prédiction)")
fig.tight_layout()
plt.show()

**Analyse :** Double-cliquez ici. Que pouvez-vous conclure sur le compromis entre le taux de rejet et la fiabilité d'un modèle d'intelligence artificielle ?

### 2.6 Régions de Prédiction Conforme (Calibration)

L'intervalle de confiance (Bootstrap) est une estimation statistique mais n'offre pas de **garantie formelle de couverture**. Si l'on exige une fiabilité de 90%, le Bootstrap peut échouer en pratique à cause d'un manque de calibration sur les données de test.

La **Prédiction Conforme (Split Conformal)** résout ce problème en utilisant un jeu de données de "Calibration" pour calculer un score de non-conformité. Pour chaque échantillon $i$ du jeu de calibration, on calcule :
$$ s_i = 1 - \hat{P}(y_i \mid x_i),$$
où $y_i$ est la classe réelle. On cherche ensuite le quantile $\hat{q}$ correspondant au niveau de confiance souhaité. Pour un nouvel individu, la **Région de Prédiction** contient toutes les classes $y$ telles que :
$$ \hat{P}(Y=y \mid X) \ge 1 - \hat{q} $$

In [ ]:
# Division du jeu d'entraînement pour créer un jeu de calibration (ex: 20%)
X_train_prop, X_cal, y_train_prop, y_cal = train_test_split(
    X_train_proc, y_train, test_size=0.2, random_state=42
)

# Ré-entraînement du modèle sur la partie 'Proper Train'
model_cp = LogisticRegression(max_iter=1000)
model_cp.fit(..., ...)

# Calcul des scores de non-conformité sur le jeu de CALIBRATION
# Le score est : 1 - (probabilité prédite pour la VRAIE classe)
probs_cal = model_cp.predict_proba(X_cal)
true_class_probs = probs_cal[np.arange(len(y_cal)), y_cal]
scores_cal = 1 - ...

print(f"Nombre d'échantillons de calibration : {len(scores_cal)}")

Nous allons maintenant évaluer si la couverture réelle (Empirique) correspond bien à la couverture demandée (Nominale), et observer l'évolution du taux d'hésitation.

In [ ]:
def evaluate_conformal(model, X_test, y_test, scores_cal, alpha_level):
    """Calcule l'accuracy de la région et le taux d'hésitation pour un alpha donné."""
    n = len(scores_cal)
    # Calcul du quantile chapeau avec correction pour échantillon fini
    q_level = np.ceil((n + 1) * (1 - alpha_level)) / n
    qhat = np.quantile(scores_cal, q_level, method='higher')
    
    # Prédiction sur le jeu de test
    probs_test = model.predict_proba(X_test)
    
    # Une classe est dans la région si sa probabilité >= 1 - qhat
    in_region_0 = probs_test[:, 0] >= (1 - qhat)
    in_region_1 = probs_test[:, 1] >= (1 - qhat)
    
    # Vérification de la couverture : la vérité est-elle dans l'ensemble prédit ?
    # Si y_test=0, on regarde in_region_0. Si y_test=1, on regarde in_region_1.
    covered = ((y_test == 0) & in_region_0) | ((y_test == 1) & in_region_1)
    accuracy_region = np.mean(covered)
    
    # Taux d'hésitation : cas où la région est {0, 1} (les deux sont True)
    hesitation = in_region_0 & ...
    hesitation_rate = np.mean(hesitation)
    
    return accuracy_region, hesitation_rate

nominal_coverages = np.linspace(0.70, 0.99, 30)
results = [evaluate_conformal(model_cp, X_test_proc, y_test, scores_cal, 1-c) for c in nominal_coverages]

empirical_acc, hesitation_rates = zip(*results)

In [ ]:
# Visualisation de la Calibration
plt.figure(figsize=(10, 6))
plt.plot(nominal_coverages * 100, empirical_acc, 'b-', lw=3, label='Précision réelle (Empirique)')
plt.plot([70, 100], [0.7, 1.0], 'k--', alpha=0.5, label='Ligne idéale (Parfaite calibration)')

plt.xlabel('Couverture Nominale Ciblée (%)')
plt.ylabel('Précision réelle de la région')
plt.title('Vérification de la Calibration (Split Conformal)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Analyse :** Comparez ce graphique avec celui de la section 2.5 (Bootstrap). Que remarquez-vous sur la position de la courbe bleue par rapport à la diagonale pointillée ? En conclusion, quel est l'avantage majeur de la Prédiction Conforme pour un système de certification (ex: aéronautique) par rapport au simple Bootstrap ?

## Partie 3 : Théorie de la Décision

La métrique AUC ne suffit pas dans le monde réel. Les erreurs n'ont pas toutes le même coût financier ou scientifique.

### 3.1 Matrice de coût (Credit-g)
Dans l'octroi de crédit, un **Faux Positif** (prêter à un mauvais payeur) coûte 5 fois plus cher qu'un **Faux Négatif** (refuser un bon client). 
Utilisez les seuils de la courbe ROC pour trouver le seuil de probabilité optimal minimisant ce coût.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, mean_prob)
couts = []

for t in thresholds:
    # Prédiction pour le seuil t
    y_pred_t = (mean_prob >= t).astype(int)
    
    # Calcul des Faux Positifs (y_test=0 et y_pred=1) et Faux Négatifs
    fp = np.sum((y_test == 0) & (y_pred_t == ...))
    fn = np.sum((... == 1) & (... == 0))
    
    # Application de la matrice de coût
    cout_total = fp * 5 + fn * 1
    couts.append(cout_total)

seuil_optimal = thresholds[np.argmin(couts)]

plt.figure(figsize=(8, 4))
plt.plot(..., ..., color='purple', lw=2)
plt.axvline(seuil_optimal, color='green', linestyle='--', label=f'Seuil Opt: {seuil_optimal:.2f}')
plt.axvline(0.5, color='gray', linestyle=':', label='Seuil standard: 0.50')
plt.xlim(0, 1)
plt.xlabel("Seuil de décision $\alpha$")
plt.ylabel("Coût Total Espéré")
plt.legend()
plt.show()

### 3.2 L'Option de Rejet de Chow (Exoplanètes)

Retournons à notre modèle Naïve Bayes sur Kepler. On introduit un astrophysicien (l'Expert) à qui l'IA peut déléguer l'étude des signaux trop incertains.
La zone d'incertitude (rejet) est définie par : $P(E \mid x) \in [0.5 - \theta, 0.5 + \theta]$.
Coût d'une erreur IA = 1. Coût de délégation = $\gamma$.

In [ ]:
thetas = np.linspace(0, 0.499, 50)
couts_experts = [0.1, 0.25, 0.4]

plt.figure(figsize=(9, 5))
colors = ['green', 'orange', 'red']

for gamma, color in zip(couts_experts, colors):
    couts_totaux = []
    for theta in thetas:
        borne_inf, borne_sup = 0.5 - theta, 0.5 + theta
        
        # Masque booléen : True si la probabilité est dans la zone d'indécision
        rejetes = (y_prob_kep >= ...) & (y_prob_kep <= ...)
        
        # Prédictions fermes (hors zone de rejet)
        y_pred_ia = (y_prob_kep > 0.5).astype(int)
        erreurs_ia = (y_pred_ia != y_kep) & (~rejetes)
        
        # Coût = (Erreurs * 1) + (Délégation * gamma)
        cout = np.sum(erreurs_ia) * 1.0 + np.sum(rejetes) * gamma
        couts_totaux.append(cout)
        
    opt_theta = thetas[np.argmin(couts_totaux)]
    plt.plot(thetas, couts_totaux, color=color, label=f'Coût Expert $\gamma$={gamma} (Opt: $\theta$={opt_theta:.2f})')
    plt.axvline(opt_theta, color=color, linestyle=':', alpha=0.7)

plt.xlabel("Largeur de la zone d'indécision IA ($\theta$)")
plt.ylabel("Coût Total (Erreurs IA + Honoraires Expert)")
plt.legend()
plt.show()

**Analyse :** Double-cliquez ici. Que constatez-vous sur l'évolution de la zone de rejet ($\theta$ optimal) lorsque l'expert coûte de plus en plus cher ($\gamma$ augmente) ?

**Note**: Bien que le seuil $\theta$ minimise le coût statistique, il faut prendre en compte **l'intrusivité** du système. En aéronautique, si l'IA délègue trop souvent à l'expert (ici l'astrophysicien), cela peut entraîner une surcharge cognitive ou une désensibilisation aux alertes ('alarm fatigue'). Le choix de $\theta$ est donc un compromis entre performance pure et acceptabilité par l'opérateur humain.